# Setup

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import random
from matplotlib import pyplot as plt
import tensorflow as tf
from numpy.typing import NDArray
from pathlib import Path

RANDOM_SEED = 42

sys.path.append(os.path.abspath("."))


2025-09-03 19:50:19.027346: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-03 19:50:19.027400: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-03 19:50:19.028410: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-03 19:50:19.109657: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def seed_random():
    np.random.seed(RANDOM_SEED)
    tf.random.set_seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)

In [3]:
seed_random()

In [5]:
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

VALID_SIZE = 10000

X_valid = X_train_full[:VALID_SIZE]
y_valid = y_train_full[:VALID_SIZE]

X_train = X_train_full[VALID_SIZE:]
y_train = y_train_full[VALID_SIZE:]

In [4]:
X_train.shape

(40000, 32, 32, 3)

In [5]:
X_test.shape

(10000, 32, 32, 3)

In [6]:
# Use smaller subsets while searching to make the whole process faster.
X_train_s = X_train[:10000]
y_train_s = y_train[:10000]

# Initial Model + Compare

- DNN with `20` hidden layers + `100` neurons on each layer
- `He` initialisation + `Swish` activation funcion
- `Nadam` optimisation
- early stopping.

In [7]:
tf.keras.backend.clear_session()

def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=[32, 32, 3])
    ])

    for _ in range(20):

        # He initialisation + Swish activation function
        layer = tf.keras.layers.Dense(
            100,
            activation="swish",
            kernel_initializer="he_normal"
        )

        model.add(layer)

    # output layer
    model.add(tf.keras.layers.Dense(10, activation="softmax"))  # according to the dataset, there's 10 possible values. Hence 10 + softmax

    return model

In [53]:
create_model().summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 3072)              0         
                                                                 
 dense (Dense)               (None, 100)               307300    
                                                                 
 dense_1 (Dense)             (None, 100)               10100     
                                                                 
 dense_2 (Dense)             (None, 100)               10100     
                                                                 
 dense_3 (Dense)             (None, 100)               10100     
                                                                 
 dense_4 (Dense)             (None, 100)               10100     
                                                                 
 dense_5 (Dense)             (None, 100)               1

I went through the same values manually here from the official solution:

- `1e-5`
- `3e-5`
- `1e-4`
- `3e-4`
- `1e-3`
- `3e-3`
- `1e-2`

Though I personally guess if I had to do this from scratch, I'd have tried to figure it out using one of the Learning Rate Schedules, probably the recommended default one (Performance Scheduling) initially.

Also, it kind of goes against the idea of starting with a large learning rate and decreasing it :D 

In [ ]:
earlystop_cb = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "cifar10_model.keras",
    monitor='val_accuracy',
    mode='max',
    #save_best_only=True  # <- (kaput for the same reason as in 10_mlp.ipynb)
)

In [ ]:
for lr in [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]:
    run_logdir = Path() / "cifar10" / f"run_{lr:.0e}"

    tf.keras.backend.clear_session()

    model = create_model()
    model.compile(
        loss="sparse_categorical_crossentropy",  # <- we are still dealing with sparse categorical data, just like the MNIST dataset
        optimizer=tf.keras.optimizers.Nadam(learning_rate=lr),  # Adam + Nesterov trick
        metrics=["accuracy"]
    )

    tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)

    history = model.fit(
        X_train,
        y_train,
        epochs=10,
        validation_data=(X_valid, y_valid),
        callbacks=[
            earlystop_cb,
            tensorboard_cb,
            checkpoint_cb
        ]
    )

Epoch 1/10
1250/1250 [==============================] - 21s 13ms/step - loss: 15.6705 - accuracy: 0.1131 - val_loss: 4.0413 - val_accuracy: 0.1116
Epoch 2/10
1250/1250 [==============================] - 19s 15ms/step - loss: 3.1792 - accuracy: 0.1310 - val_loss: 2.7516 - val_accuracy: 0.1267
Epoch 3/10
1250/1250 [==============================] - 17s 13ms/step - loss: 2.5284 - accuracy: 0.1407 - val_loss: 2.3946 - val_accuracy: 0.1437
Epoch 4/10
1250/1250 [==============================] - 19s 15ms/step - loss: 2.2966 - accuracy: 0.1757 - val_loss: 2.2139 - val_accuracy: 0.1927
Epoch 5/10
1250/1250 [==============================] - 20s 16ms/step - loss: 2.1657 - accuracy: 0.2067 - val_loss: 2.1262 - val_accuracy: 0.2148
Epoch 6/10
1250/1250 [==============================] - 17s 13ms/step - loss: 2.0771 - accuracy: 0.2352 - val_loss: 2.0565 - val_accuracy: 0.2418
Epoch 7/10
1250/1250 [==============================] - 18s 14ms/step - loss: 2.0105 - accuracy: 0.2591 - val_loss: 1.9911 

Here were my TensorBoard reults for the proposed values:

![My Tensorflow Results](./2025-08-31_22-24.png)

Again I think per the suggestions the idea would've been to actually start at high values and just use one of the strategies mentioned on p.389 (Performance scheduling being the default suggested strategy) but I decided to recreate the setup that Geron discussed him using to see what he might've seen and how the optimal selection would've looked.

In [5]:
GOOD_LEARNING_RATE = 5e-5

In [ ]:
# So, Geron mentions he ended up testing 5e-5 too and it turning out slightly better. Let's see that then
tf.keras.backend.clear_session()

model = create_model()
model.compile(
    loss="sparse_categorical_crossentropy",  # <- we are still dealing with sparse categorical data, just like the MNIST dataset
    optimizer=tf.keras.optimizers.Nadam(learning_rate=GOOD_LEARNING_RATE),  # Adam + Nesterov trick
    metrics=["accuracy"]
)

tensorboard_cb = tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"run_{GOOD_LEARNING_RATE:.0e}")

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_valid, y_valid),
    callbacks=[
        earlystop_cb,
        tensorboard_cb,
        checkpoint_cb
    ]
)

Epoch 1/10
1250/1250 [==============================] - 19s 12ms/step - loss: 9.0002 - accuracy: 0.1525 - val_loss: 2.2011 - val_accuracy: 0.1992
Epoch 2/10
1250/1250 [==============================] - 19s 15ms/step - loss: 2.1151 - accuracy: 0.2267 - val_loss: 2.0733 - val_accuracy: 0.2363
Epoch 3/10
1250/1250 [==============================] - 15s 12ms/step - loss: 2.0022 - accuracy: 0.2639 - val_loss: 1.9703 - val_accuracy: 0.2861
Epoch 4/10
1250/1250 [==============================] - 19s 15ms/step - loss: 1.9266 - accuracy: 0.2919 - val_loss: 1.8663 - val_accuracy: 0.3122
Epoch 5/10
1250/1250 [==============================] - 15s 12ms/step - loss: 1.8750 - accuracy: 0.3151 - val_loss: 1.8625 - val_accuracy: 0.3193
Epoch 6/10
1250/1250 [==============================] - 19s 15ms/step - loss: 1.8246 - accuracy: 0.3350 - val_loss: 1.8083 - val_accuracy: 0.3487
Epoch 7/10
1250/1250 [==============================] - 15s 12ms/step - loss: 1.7804 - accuracy: 0.3555 - val_loss: 1.7491 -

![My results of 5e-5](./2025-08-31_22-33.png)

So let's go with `5e-5` for now, even though in my case I think in my case `1e-4` is the better option.

In [ ]:
tf.keras.backend.clear_session()

model = create_model()
model.compile(
    loss="sparse_categorical_crossentropy",  # <- we are still dealing with sparse categorical data, just like the MNIST dataset
    optimizer=tf.keras.optimizers.Nadam(learning_rate=GOOD_LEARNING_RATE),  # Adam + Nesterov trick
    metrics=["accuracy"]
)

earlystop_cb = tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)
tensorboard_cb = tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"run_{GOOD_LEARNING_RATE:.0e}")

# fit for 100 epochs!
history = model.fit(
    X_train,
    y_train,
    epochs=100, # <- full run this time, not just 10 epochs as during the learning rate finding...
    validation_data=(X_valid, y_valid),
    callbacks=[
        earlystop_cb,
        tensorboard_cb,
        checkpoint_cb
    ]
)

2025-08-31 23:17:11.073804: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.080101: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.080125: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.081468: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.081503: I tensorflow/compile

Epoch 1/100


2025-08-31 23:17:13.795492: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x745cc1e06840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-08-31 23:17:13.795522: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2025-08-31 23:17:13.801806: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-08-31 23:17:13.809153: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2025-08-31 23:17:13.868148: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1250/1250 [==============================] - 21s 14ms/step - loss: 6.6086 - accuracy: 0.1543 - val_loss: 2.2072 - val_accuracy: 0.1988
Epoch 2/100
1250/1250 [==============================] - 16s 13ms/step - loss: 2.1276 - accuracy: 0.2188 - val_loss: 2.0919 - val_accuracy: 0.2321
Epoch 3/100
1250/1250 [==============================] - 17s 14ms/step - loss: 2.0183 - accuracy: 0.2566 - val_loss: 1.9827 - val_accuracy: 0.2682
Epoch 4/100
1250/1250 [==============================] - 15s 12ms/step - loss: 1.9324 - accuracy: 0.2889 - val_loss: 1.8801 - val_accuracy: 0.3089
Epoch 5/100
1250/1250 [==============================] - 19s 15ms/step - loss: 1.8636 - accuracy: 0.3176 - val_loss: 1.8242 - val_accuracy: 0.3313
Epoch 6/100
1250/1250 [==============================] - 15s 12ms/step - loss: 1.8040 - accuracy: 0.3434 - val_loss: 1.7839 - val_accuracy: 0.3403
Epoch 7/100
1250/1250 [==============================] - 18s 14ms/step - loss: 1.7558 - accuracy: 0.3630 - val_loss: 1.7166 - val_

Bonus visualisation of remote WSL2 VM, just to see what it's actually up to as it's converging...

```text

Every 2.0s: nvidia-smi                                                                                                                                                                                                                                                                      shane: Sun Aug 31 22:41:49 2025

Sun Aug 31 22:41:49 2025
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.76.07              Driver Version: 581.08         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   57C    P5             14W /   79W |    9736MiB /  12282MiB |     38%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|    0   N/A  N/A          343769      C   /python3.10                           N/A      |
+-----------------------------------------------------------------------------------------+
```

For me the whole processing is surprisingly taking longer than for Geron. 

In [ ]:
model.evaluate(X_valid, y_valid) # [loss, accuracy]

313/313 [==============================] - 1s 3ms/step - loss: 1.4890 - accuracy: 0.4823


[1.4889771938323975, 0.4823000133037567]

More or less similar result, I guess. `48.2%` accuracy. Our best model got saved to `cifar10_model.keras`

# Compare with Batch Normalisation vs no Batch Normalisation

Adding Batch Normalisation + compare

In [15]:
def create_model_bn():

    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=[32, 32, 3]),
        tf.keras.layers.BatchNormalization() # <- we need it after every layer except the last one
    ])

    for _ in range(20):
        model.add(tf.keras.layers.Dense(
            100,
            #activation="swish", # (see below)
            kernel_initializer="he_normal",
            use_bias=False # <- see below
        ))

        # Ok so I added mine after the activation function, but I see Geron's solution puts it before.
        # And I'd like to try out that solution to kind of "work along" with it. So I'll copy taht setup here
        #model.add(tf.keras.layers.BatchNormalization()) # <- we need it after every layer except the last one

        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation("swish"))

    # output layer
    model.add(tf.keras.layers.Dense(10, activation="softmax"))

    return model

In [18]:
tf.keras.backend.clear_session()
seed_random()

CHECKPOINT_FILE_NAME = "cifar10_model_bn.keras"
GOOD_LEARNING_RATE = 5e-4 # <- updated to Geron's find. See below.

batchnorm_model = create_model_bn()
batchnorm_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Nadam(learning_rate=GOOD_LEARNING_RATE),
    metrics=["accuracy"]
)

# fit for 100 epochs!
history = batchnorm_model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(X_valid, y_valid),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"full_run_{GOOD_LEARNING_RATE:.0e}_batchnorm"),
        tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_FILE_NAME, monitor='val_accuracy', mode='max')
    ]
)

Epoch 1/100
1250/1250 [==============================] - 37s 25ms/step - loss: 2.0711 - accuracy: 0.2399 - val_loss: 1.8774 - val_accuracy: 0.3282
Epoch 2/100
1250/1250 [==============================] - 30s 24ms/step - loss: 1.8166 - accuracy: 0.3415 - val_loss: 1.6815 - val_accuracy: 0.3863
Epoch 3/100
1250/1250 [==============================] - 31s 25ms/step - loss: 1.7079 - accuracy: 0.3870 - val_loss: 1.6061 - val_accuracy: 0.4254
Epoch 4/100
1250/1250 [==============================] - 32s 26ms/step - loss: 1.6338 - accuracy: 0.4142 - val_loss: 1.5527 - val_accuracy: 0.4421
Epoch 5/100
1250/1250 [==============================] - 33s 26ms/step - loss: 1.5858 - accuracy: 0.4375 - val_loss: 1.4955 - val_accuracy: 0.4612
Epoch 6/100
1250/1250 [==============================] - 32s 26ms/step - loss: 1.5365 - accuracy: 0.4528 - val_loss: 1.4809 - val_accuracy: 0.4714
Epoch 7/100
1250/1250 [==============================] - 32s 25ms/step - loss: 1.4942 - accuracy: 0.4692 - val_loss: 1

In [19]:
batchnorm_model.evaluate(X_valid, y_valid) # [loss, accuracy]

313/313 [==============================] - 2s 5ms/step - loss: 1.3530 - accuracy: 0.5291


[1.3530347347259521, 0.5291000008583069]

We'll answer these questions from the book:

1. Is it converging faster than before?

2. Does it produce a better model?

3. How does it affect training speed?

-------

### Results:

<details>
<summary>OLD</summary>

So first I ended up just running it with the original learning rate that we determined in pt1 of `5e-5`.
Then I saw Geron's remark regarding running dedicated learning rate investigations for this part too, which makes sense with the whole "try to find a good learning rate every time you change the params".

Now my GPU is being weirdly slow for some reason, slower than Geron's reported time, which makes the whole thing slow. But had I done this again, I would've either used one of the Learning Rate Schedules suggested from the book, probably Performance scheduling by default. And if not that, just reused that comparison loop I created in pt1 above with the new model.

But I ended up goiing with the `5e-4` that Geron found for lack of time here.

![Tensorflow curves comparison Batch norm vs no batch norm](./2025-09-01_10-40.png)
</details>

![Tensorflow curves comparison Batch norm vs no batch norm - run 2 with learning rate 5e-4](./2025-09-01_11-14.png)

**To the Questions:**

1. Yes, it's converging faster than before. See both the Train and Validation curves above.

2. It does produce a better model. The Validation accuracy is `52.9%` as compared to the last `48.2%`

3. After around epoch 7, the training speed is clearly better. However, @ around epoch 25, we stop really seeing improvements in the accuracy on the Validation set (it seems to flatline). Probably starting to overfit the Train set. 
   
   **Note regarding seeing the official solution notes:** 
   And yeah, every epoch is also taking longer. I think my average was maybe 17s before, whereas here it seems to hit the 30s. That's even worse than what Geron reports with his CPU (and makes me really confused why the 4080 isn't helping me at all)


_Notes/takeaways for now is to look into why on earth the GPU is underperforming relative to Geron's reported times... eventually_

# Compare against SELU with other adjustments

*Exercise: Try replacing Batch Normalization with SELU, and make the necessary adjustements to ensure the network self-normalizes (i.e., standardize the input features, use LeCun normal initialization, make sure the DNN contains only a sequence of dense layers, etc.).*

In [25]:
def create_model_selu(all_axis: bool = False):

    #self-normalising network
    normalisation = tf.keras.layers.Normalization(
        input_shape=[32, 32, 3],
        # by default: computes against JUST the last index -- aka group by RGB!
        axis=(-1 if not all_axis else (1, 2, 3)) # https://keras.io/api/layers/preprocessing_layers/numerical/normalization/
        # ^ explicitly specifying (1, 2, 3), this will ensure every multi-dimensional value gets separated properly and computed properly
    )

    model = tf.keras.Sequential([
        normalisation,
        tf.keras.layers.Flatten(input_shape=[32, 32, 3])
    ])

    for _ in range(20):
        model.add(tf.keras.layers.Dense(
            100,
            activation="selu",
            kernel_initializer="lecun_normal"
        ))

    # output layer
    model.add(tf.keras.layers.Dense(10, activation="softmax"))

    return (model, normalisation)

In [ ]:
CHECKPOINT_FILE_NAME = "cifar10_model_selu.keras"

# here I'll do the full loop to determine a good learning rate!
for lr in [1e-5, 3e-5, 1e-4, 3e-4, 7e-4, 1e-3, 3e-3, 1e-2]:
    tf.keras.backend.clear_session()
    seed_random()

    selu_model, normalisation = create_model_selu()
    selu_model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=tf.keras.optimizers.Nadam(learning_rate=lr),
        metrics=["accuracy"]
    )
    normalisation.adapt(X_train) # <- required before fit() if we normalise by using the normalisation layer

    history = selu_model.fit(
        X_train,
        y_train,
        epochs=10, # <- just 10
        validation_data=(X_valid, y_valid),
        callbacks=[
            tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
            tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"run_selu_{lr:.0e}"),
            #tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_FILE_NAME, monitor='val_accuracy', mode='max')
        ]
    )

Epoch 1/10
1250/1250 [==============================] - 21s 13ms/step - loss: 1.9515 - accuracy: 0.3030 - val_loss: 1.7946 - val_accuracy: 0.3586
Epoch 2/10
1250/1250 [==============================] - 16s 13ms/step - loss: 1.7412 - accuracy: 0.3816 - val_loss: 1.7243 - val_accuracy: 0.3795
Epoch 3/10
1250/1250 [==============================] - 15s 12ms/step - loss: 1.6432 - accuracy: 0.4181 - val_loss: 1.6654 - val_accuracy: 0.4085
Epoch 4/10
1250/1250 [==============================] - 17s 14ms/step - loss: 1.5790 - accuracy: 0.4396 - val_loss: 1.5923 - val_accuracy: 0.4283
Epoch 5/10
1250/1250 [==============================] - 16s 13ms/step - loss: 1.5229 - accuracy: 0.4625 - val_loss: 1.5543 - val_accuracy: 0.4530
Epoch 6/10
1250/1250 [==============================] - 16s 13ms/step - loss: 1.4733 - accuracy: 0.4822 - val_loss: 1.6032 - val_accuracy: 0.4583
Epoch 7/10
1250/1250 [==============================] - 17s 13ms/step - loss: 1.4312 - accuracy: 0.4978 - val_loss: 1.5316 -

I tweaked the learning rates in my loop from before to include Geron's `7e-4` and used the keras `Normalisation` layer here instead of manually tweaking the standardised values. 

Result:

![Train set curves for SELU setup](./2025-09-03_21-10_1.png)

![Validation set curves for SELU setup](./2025-09-03_21-10.png)

In [26]:
CHECKPOINT_FILE_NAME = "cifar10_model_selu.keras"
GOOD_LEARNING_RATE = 7e-4

tf.keras.backend.clear_session()
seed_random()

selu_model, normalisation = create_model_selu(all_axis=True)
selu_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Nadam(learning_rate=GOOD_LEARNING_RATE),
    metrics=["accuracy"]
)
normalisation.adapt(X_train) # <- I think normalising using the layer is a bit less messy than doing it manually? *shrug*

history = selu_model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(X_valid, y_valid),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"full_run_{GOOD_LEARNING_RATE:.0e}_selu_v2"),
        tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_FILE_NAME, monitor='val_accuracy', mode='max')
    ]
)

Epoch 1/100
1250/1250 [==============================] - 20s 13ms/step - loss: 1.9761 - accuracy: 0.2886 - val_loss: 1.7924 - val_accuracy: 0.3630
Epoch 2/100
1250/1250 [==============================] - 17s 14ms/step - loss: 1.7499 - accuracy: 0.3779 - val_loss: 1.7233 - val_accuracy: 0.3801
Epoch 3/100
1250/1250 [==============================] - 16s 13ms/step - loss: 1.6451 - accuracy: 0.4206 - val_loss: 1.6447 - val_accuracy: 0.4135
Epoch 4/100
1250/1250 [==============================] - 17s 14ms/step - loss: 1.5702 - accuracy: 0.4444 - val_loss: 1.5865 - val_accuracy: 0.4355
Epoch 5/100
1250/1250 [==============================] - 16s 13ms/step - loss: 1.5174 - accuracy: 0.4669 - val_loss: 1.5865 - val_accuracy: 0.4432
Epoch 6/100
1250/1250 [==============================] - 17s 14ms/step - loss: 1.4596 - accuracy: 0.4873 - val_loss: 1.5479 - val_accuracy: 0.4696
Epoch 7/100
1250/1250 [==============================] - 16s 13ms/step - loss: 1.4238 - accuracy: 0.5000 - val_loss: 1

In [27]:
selu_model.evaluate(X_valid, y_valid)

313/313 [==============================] - 1s 4ms/step - loss: 1.4788 - accuracy: 0.5079


[1.4788082838058472, 0.5078999996185303]

So I actually ended up doing two versions of the same thing here. 

Because my solution wasn't the same as far as Normalisation goes. So first of all I started off having the `Normalization()` layer default to `axis=-1` which caused the Normalisation layer to group values together by RGB values and not every `32 * 32 * 3` pixel being normalised by itself. This gave a final validation accuracy of `49.x%`. 

I then figured out that this was not the same as Geron's solution, which is computing a separate mean and standard deviation for each of the `32 * 32 * 3` pixels manually, and my setup above with `all_axis=True` should more or less replicate the mean and stdev being computed per individual pixel channel. The validation accuracy of *that* model ended up slightly better at `50.8%`.

For the sake of curiosity, here are the curves for both the train and validation sets, compared to the Batch Normalisation setup and the first attempt with no tweaks:


Train Set:
![Train Set SELU comparison](./2025-09-03_22-16_1.png)

Validation Set:
![Validation Set SELU comparison](./2025-09-03_22-16.png)

Nonetheless, the results here are the same as in the official solution. SELU converged faster than the previous two, and the validation accuracy was lied between the two. Interestingly the train curves look better which I guess indicates overfitting for the train data. Oh and it was faster to train per epoch, too. (But I am still confused why my overpriced GPU is doing worse than Geron's CPU 😅)

# Alpha Dropout + MC Dropout

In [28]:
import keras.layers

def create_model_selu_alphadropout(spam_dropout: bool, dropout_prob: float):
    #self-normalising network + dropout (we have to use AlphaDropout with SELU)

    normalisation = tf.keras.layers.Normalization(
        input_shape=[32, 32, 3],
        axis=(1, 2, 3)
    )

    # okay so apparently `tf.keras.layers.AlphaDropout()` is bugged per the solution
    # so I'll use the one from the sidebar.
    # ALSO apparently the idea is that you can use AlphaDropout sparingly.
    # You don't need to spam it on every layer. Hence why the canonical solution only
    # has it before the final layer.
    # but... I am curious to see what the effect of spamming it vs not spamming it is.
    # So I'll test this.

    model = tf.keras.Sequential([
        normalisation,
        tf.keras.layers.Flatten(input_shape=[32, 32, 3])
    ])

    for _ in range(20):
        if spam_dropout:
            model.add(keras.layers.AlphaDropout(rate=dropout_prob))

        model.add(tf.keras.layers.Dense(
            100,
            activation="selu",
            kernel_initializer="lecun_normal"
        ))

    model.add(keras.layers.AlphaDropout(rate=dropout_prob))

    # output layer
    model.add(tf.keras.layers.Dense(10, activation="softmax"))

    return (model, normalisation)

In [30]:
#CHECKPOINT_FILE_NAME = "cifar10_model_selu_alphadropout.keras"

for lr in [3e-5, 1e-4, 3e-4, 5e-4, 7e-4]:
    for do_spam in [True, False]:
        tf.keras.backend.clear_session()
        seed_random()

        selu_ad_model, normalisation = create_model_selu_alphadropout(do_spam, dropout_prob=0.1) # <- the 0.1 is stolen from the official solution.
        selu_ad_model.compile(
            loss="sparse_categorical_crossentropy",
            optimizer=tf.keras.optimizers.Nadam(learning_rate=lr),
            metrics=["accuracy"]
        )
        normalisation.adapt(X_train) # <- required before fit() if we normalise by using the normalisation layer

        history = selu_ad_model.fit(
            X_train,
            y_train,
            epochs=10, # <- just 10
            validation_data=(X_valid, y_valid),
            callbacks=[
                tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
                tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"run_selu_ad_{lr:.0e}_{do_spam}"),
                #tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_FILE_NAME, monitor='val_accuracy', mode='max')
            ]
        )

Epoch 1/10
1250/1250 [==============================] - 23s 15ms/step - loss: 2.7818 - accuracy: 0.1212 - val_loss: 2.7259 - val_accuracy: 0.1837
Epoch 2/10
1250/1250 [==============================] - 18s 14ms/step - loss: 2.4861 - accuracy: 0.1551 - val_loss: 2.4349 - val_accuracy: 0.1502
Epoch 3/10
1250/1250 [==============================] - 18s 15ms/step - loss: 2.3929 - accuracy: 0.1558 - val_loss: 2.4399 - val_accuracy: 0.1778
Epoch 4/10
1250/1250 [==============================] - 17s 14ms/step - loss: 2.3216 - accuracy: 0.1600 - val_loss: 2.4124 - val_accuracy: 0.1740
Epoch 5/10
1250/1250 [==============================] - 19s 15ms/step - loss: 2.2605 - accuracy: 0.1639 - val_loss: 2.3514 - val_accuracy: 0.1740
Epoch 6/10
1250/1250 [==============================] - 19s 15ms/step - loss: 2.2213 - accuracy: 0.1634 - val_loss: 2.3067 - val_accuracy: 0.1707
Epoch 7/10
1250/1250 [==============================] - 17s 14ms/step - loss: 2.1761 - accuracy: 0.1699 - val_loss: 2.2096 -

So first of all, I learned that apparently the idea is that spamming `AlphaDropout` on every layer like the general Dropout example is  not the idea. Hence Geron's solution only has it before the final output layer.

Here's why (comparing training rates where the green lines represent the curves for only including the dropout on the last layer, and the pink lines represent the curves for including it on every layer as with the regular dropout examples we saw before):

![Curves for AlphaDroput on every layer versus only after final hidden layer](./2025-09-03_23-21.png)

Also I should've just used the HyperParameter search setup at this point + the HPARAMS tab, but didn't think of this immediately, hence I'm stuck in this weird format of reinventing Hyperparameter search for 2 params from scratch in this Time Series tab. Whoops.

Anyways, for me it actually looks like these are the best options from the 10 epochs: learning rate `3e-4` and `1e-4`:
![Best SELU + Alpha Dropout learning rates](./2025-09-03_23-27.png)

I'll just go with `3e-4` to keep the solution a bit more close to Geron's for now. We'll do a full 100 epoch train loop with early stopping using this now.

In [32]:
CHECKPOINT_FILE_NAME = "cifar10_model_selu_alphadropout.keras"
GOOD_LEARNING_RATE = 3e-4

tf.keras.backend.clear_session()
seed_random()

selu_ad_model, normalisation = create_model_selu_alphadropout(False, dropout_prob=0.1) # <- the 0.1 is stolen from the official solution.
selu_ad_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Nadam(learning_rate=GOOD_LEARNING_RATE),
    metrics=["accuracy"]
)
normalisation.adapt(X_train) # <- required before fit() if we normalise by using the normalisation layer

history = selu_ad_model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(X_valid, y_valid),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"full_run_{GOOD_LEARNING_RATE:.0e}_selu_alpha_dropout"),
        tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_FILE_NAME, monitor='val_accuracy', mode='max')
    ]
)

Epoch 1/100
1250/1250 [==============================] - 21s 14ms/step - loss: 1.9375 - accuracy: 0.3152 - val_loss: 1.7268 - val_accuracy: 0.3849
Epoch 2/100
1250/1250 [==============================] - 16s 13ms/step - loss: 1.6696 - accuracy: 0.4089 - val_loss: 1.6589 - val_accuracy: 0.4193
Epoch 3/100
1250/1250 [==============================] - 18s 14ms/step - loss: 1.5645 - accuracy: 0.4471 - val_loss: 1.6037 - val_accuracy: 0.4313
Epoch 4/100
1250/1250 [==============================] - 16s 12ms/step - loss: 1.4845 - accuracy: 0.4749 - val_loss: 1.5398 - val_accuracy: 0.4702
Epoch 5/100
1250/1250 [==============================] - 17s 14ms/step - loss: 1.4267 - accuracy: 0.4967 - val_loss: 1.5389 - val_accuracy: 0.4601
Epoch 6/100
1250/1250 [==============================] - 15s 12ms/step - loss: 1.3706 - accuracy: 0.5167 - val_loss: 1.5434 - val_accuracy: 0.4938
Epoch 7/100
1250/1250 [==============================] - 17s 14ms/step - loss: 1.3229 - accuracy: 0.5350 - val_loss: 1

In [33]:
selu_ad_model.evaluate(X_valid, y_valid)

313/313 [==============================] - 1s 4ms/step - loss: 1.4891 - accuracy: 0.4923


[1.489147663116455, 0.49230000376701355]

So interestingly, just as with Geron's implementation, the final validation result is not much better. The accuracy on the train set is quite high, though, compared to the others, so I guess it's overfitting that mostly:

![SELU with Alpha Dropout full curve Train](./2025-09-03_23-45.png)

![SELU with Alpha Dropout full curve Validation](./2025-09-03_23-45_1.png)

The curiosity here is which boost we will get from Monte Carlo Dropout, I guess! 🤔

In [35]:
# the wacky override hack classs (but this time for Alpha Dropout)
# (I am still outrageously amused that this is the canonical solution to do this in the library)

class MCAlphaDropout(keras.layers.AlphaDropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

I'm not sure if I love this idea of class construction, but I guess I can live with it... (It's a bit hard to follow imo).

In [60]:
selu_ad_mc_model = tf.keras.Sequential([
    MCAlphaDropout(l.rate) if isinstance(l, keras.layers.AlphaDropout) else l
    for l in selu_ad_model.layers
])
#selu_ad_mc_model.set_weights(selu_ad_model.get_weights())

So personally I would be tempted to just wrap the model in a class at this point to support the Monte Carlo predictions.

I like the class-based construction over utility functions, because it has a nice tendency to group functionality together and also make it obvious "what you can do with things". But this is a purely software design thing opinion. 

In [62]:
class SeluMCDropoutModel:
    def __init__(self, model: keras.Model):
        self._model = model

    def predict_probas(self, X: NDArray[np.uint8], n_samples: int = 10):
        Y_probas: list[NDArray[np.float32]] = [self._model.predict(X) for _ in range(n_samples)]
        return np.mean(Y_probas, axis=0)

    def predict_classes(self, X: NDArray[np.uint8], n_samples: int = 10) -> NDArray[np.int64]:
        Y_probas: NDArray[np.float32] = self.predict_probas(X, n_samples)
        return Y_probas.argmax(axis=1)


In [ ]:
seed_random()

# increasing n_samples slightly increases the accuracy here...
y_pred = SeluMCDropoutModel(selu_ad_mc_model).predict_classes(X_valid, n_samples=30)
y_pred

313/313 [==============================] - 1s 2ms/step


array([5, 9, 9, ..., 9, 1, 7])

In [73]:
y_valid[:, 0]

array([6, 9, 9, ..., 1, 1, 5], dtype=uint8)

In [74]:
accuracy = (y_pred == y_valid[:, 0]).mean()
accuracy

0.4923

It's interesting that in my case I can't get the accuracy up from what I had without the Monte Carlo. I think Geron's model with 10 samples didn't really make that much of a win, but in my case the Monte Carlo model is slightly underperforming the non-MC one! 🤯